# HMM (Hidden Markov Model) smoothing (forward-backward)

_Artificial Intelligence — more_

**Use the future as well as the past. Combine both directions for the best estimate of each state.**

Filtering an HMM uses only past clues. But to estimate a past hidden state, the clues that came after it also help.

     The forward-backward algorithm runs two passes. The forward pass $\alpha$ gathers evidence from the start up to step $i$; the backward pass $\beta$ gathers evidence from the end back to step $i$.

---

This notebook is a practice scaffold for the **HMM (Hidden Markov Model) smoothing (forward-backward)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Define the HMMAn HMM is fully described by three pieces: the start distribution `pi` over hidden states, the transition matrix `T` where `T[i, j] = P(state j | state i)`, and the emission matrix `E` where `E[:, o]` gives `P(observation o | state)`. We use two hidden states and a short observed sequence `obs`.

In [ ]:
import numpy as np

# 2 hidden states.
pi = np.array([0.6, 0.4])                 # start distribution over states
T  = np.array([[0.7, 0.3],                # T[i,j] = P(state j | state i)
               [0.4, 0.6]])
E  = np.array([[0.9, 0.1],                # state 0 emission probs for obs {0,1}
               [0.2, 0.8]])               # state 1 emission probs for obs {0,1}

obs = [0, 1, 0]                           # the observed sequence
N = len(obs)

### Step 2 — Run the forward and backward passesThe forward pass fills `alpha`: `alpha[i]` accumulates the evidence from the start up to step `i`. We initialise it with the start distribution times the first emission, then propagate forward by multiplying through `T` and weighting by each new observation. The backward pass fills `beta`: `beta[i]` accumulates the evidence from the end back to step `i`, starting from all-ones at the last step and propagating backward.

In [ ]:
alpha = np.zeros((N, 2))
beta = np.zeros((N, 2))

# Forward pass: evidence from the start up to step i.
alpha[0] = pi * E[:, obs[0]]                       # forward init
for i in range(1, N):
    alpha[i] = (alpha[i-1] @ T) * E[:, obs[i]]

# Backward pass: evidence from the end back to step i.
beta[N-1] = 1.0                                    # backward init
for i in range(N-2, -1, -1):
    beta[i] = T @ (E[:, obs[i+1]] * beta[i+1])

### Step 3 — Combine both directions into smoothed beliefsSmoothing multiplies the two passes together: `alpha * beta` fuses the evidence from before and after each step. Normalising each row so it sums to 1 turns these into a proper probability distribution — the smoothed posterior `P(H_i | E)` over the hidden state at every step, using **all** observations rather than just the past.

In [ ]:
# Fuse forward and backward evidence at every step.
post = alpha * beta
post /= post.sum(axis=1, keepdims=True)            # smoothed P(H_i | E)

print("smoothed posteriors per step:")
print(np.round(post, 3))

## Visualize the data & results

_Umbrella world: a guard sees the boss carry an umbrella on days 1,2,4,5 but not day 3 — what is the smoothed belief that it rained each day?_

### Step 1 — Set up the umbrella-world HMMThis is the classic umbrella world (Russell & Norvig). The hidden state each day is the weather: no rain or rain. Weather tends to **persist** from day to day (that's the transition matrix), and an umbrella is far more likely on a rainy day (that's the emission matrix). The guard observes an umbrella on days 1, 2, 4, 5 but not on day 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Umbrella-world HMM (Russell & Norvig). States: [no rain, rain].
pi = np.array([0.5, 0.5])
T  = np.array([[0.7, 0.3],                # weather persists day to day
               [0.3, 0.7]])
E  = np.array([[0.8, 0.2],                # no rain: umbrella unlikely
               [0.1, 0.9]])               # rain:    umbrella likely

obs = [1, 1, 0, 1, 1]                      # umbrella on days 1,2,4,5; none on day 3
n = len(obs)

### Step 2 — Run forward-backward over the five daysThis is exactly the same two-pass algorithm as the reference implementation, now applied to the five-day umbrella sequence. The forward pass gathers evidence up to each day, the backward pass gathers evidence after each day, and multiplying then normalising gives the smoothed belief that it rained on each day.

In [ ]:
alpha = np.zeros((n, 2))
beta = np.zeros((n, 2))

# Forward pass.
alpha[0] = pi * E[:, obs[0]]
for i in range(1, n):
    alpha[i] = (alpha[i-1] @ T) * E[:, obs[i]]

# Backward pass.
beta[n-1] = 1.0
for i in range(n-2, -1, -1):
    beta[i] = T @ (E[:, obs[i+1]] * beta[i+1])

# Smoothed P(Rain) per day.
post = alpha * beta
post /= post.sum(axis=1, keepdims=True)
days = [1, 2, 3, 4, 5]

### Step 3 — Plot the smoothed rain beliefWe plot `post[:, 1]`, the smoothed probability of rain on each day. Notice the dip on day 3, where the missing umbrella is evidence against rain — but because the neighbouring days strongly suggest rain, smoothing keeps day 3's belief from collapsing entirely. That blending of past and future evidence is the whole point of forward-backward.

In [ ]:
fig, ax = plt.subplots()
ax.plot(days, post[:, 1], "-o", color="#4ea1ff", label="P(Rain)")

ax.set_xticks(days)
ax.set_xlabel("day")
ax.set_ylabel("P(Rain = True | all evidence)")
ax.set_title("Smoothed P(Rain | all 5 umbrella observations) per day")
ax.legend()
plt.show()